# Multilayer neural network: lab notebook

**Machine Learning (5CS4-05), Recommenders, neural networks, deep learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier

### 1. A tiny MLP from scratch: 2 inputs -> H tanh units -> 1 sigmoid output

In [ ]:
# 1. A tiny MLP from scratch: 2 inputs -> H tanh units -> 1 sigmoid output
def train_mlp(X, y, H=4, lr=0.5, epochs=3000, seed=0):
    rng = np.random.default_rng(seed)
    W1, b1 = rng.normal(0, 1, (2, H)), np.zeros(H)
    W2, b2 = rng.normal(0, 1, H), 0.0
    for epoch in range(epochs):
        h = np.tanh(X @ W1 + b1)                       # forward
        o = 1 / (1 + np.exp(-(h @ W2 + b2)))
        d2 = (o - y) / len(y)                          # gradient of cross-entropy wrt output z
        gW2, gb2 = h.T @ d2, d2.sum()
        d1 = np.outer(d2, W2) * (1 - h ** 2)           # backpropagate through tanh
        gW1, gb1 = X.T @ d1, d1.sum(axis=0)
        W1 -= lr * gW1; b1 -= lr * gb1; W2 -= lr * gW2; b2 -= lr * gb2
    h = np.tanh(X @ W1 + b1); o = 1 / (1 + np.exp(-(h @ W2 + b2)))
    loss = -np.mean(y * np.log(o + 1e-12) + (1 - y) * np.log(1 - o + 1e-12))
    return ((o >= 0.5) == y).mean(), loss

### 2. XOR needs a hidden layer

In [ ]:
# 2. XOR needs a hidden layer
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)
for H in (1, 2, 4):
    best = max((train_mlp(X, y, H=H, seed=s) for s in range(10)), key=lambda r: r[0])     # best of 10 random starts
    print(f"XOR with {H} hidden unit(s): best accuracy = {best[0]:.2f}, loss = {best[1]:.3f}")

### 3. Two moons with scikit-learn: more hidden units, more flexible boundary

In [ ]:
# 3. Two moons with scikit-learn: more hidden units, more flexible boundary
Xm, ym = make_moons(400, noise=0.2, random_state=0)
Xtr, ytr, Xte, yte = Xm[:300], ym[:300], Xm[300:], ym[300:]
for sizes in [(2,), (3,), (10,), (10, 10)]:
    clf = MLPClassifier(sizes, activation="tanh", max_iter=3000, random_state=0).fit(Xtr, ytr)
    n_par = sum(w.size for w in clf.coefs_) + sum(b.size for b in clf.intercepts_)
    print(f"hidden {str(sizes):9s} parameters = {n_par:4d}  train acc = {clf.score(Xtr, ytr):.3f}  test acc = {clf.score(Xte, yte):.3f}")

### 4. Counting parameters: layers 3 -> 4 -> 2

In [ ]:
# 4. Counting parameters: layers 3 -> 4 -> 2
print("parameters of 3-4-2:", (3 + 1) * 4 + (4 + 1) * 2)

## Expected output

Running all cells above should print:

```
XOR with 1 hidden unit(s): best accuracy = 0.75, loss = 0.478
XOR with 2 hidden unit(s): best accuracy = 1.00, loss = 0.002
XOR with 4 hidden unit(s): best accuracy = 1.00, loss = 0.002
hidden (2,)      parameters =    9  train acc = 0.843  test acc = 0.920
hidden (3,)      parameters =   13  train acc = 0.850  test acc = 0.930
hidden (10,)     parameters =   41  train acc = 0.853  test acc = 0.930
hidden (10, 10)  parameters =  151  train acc = 0.857  test acc = 0.920
parameters of 3-4-2: 26
```

## Practice exercises

Try these in the cells below.

1. Train an MLPClassifier with hidden_layer_sizes=(2,) and then (8,) on the moons data. Compare the accuracy.
2. Replace tanh by identity in the from-scratch code. Can it still learn XOR?
3. Add early stopping and print how many epochs were used.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
